# Week 4 — Day 2 — Exercises XP
## Diabetes Classification with Logistic Regression

**Dataset:** *Diabetes Prediction Dataset* (Kaggle).  
Columns expected: `gender, age, hypertension, heart_disease, smoking_history, bmi, HbA1c_level, blood_glucose_level, diabetes`.

**Pipeline**
1. Problem understanding + data collection
2. Model choice + standardization
3. Model training
4. Evaluation metrics (accuracy, precision, recall, F1, confusion matrix)
5. 2-D decision boundary visualization
6. ROC curve and AUC

All comments are in English.

In [ ]:
# Shared imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score)

sns.set_theme(style='whitegrid')
np.random.seed(42)
BASE_DIR = os.getcwd()

## Exercise 1 — Problem Understanding and Data Collection

Goal: predict whether an individual has **diabetes** (binary classification).  
Tasks:
- Load the diabetes dataset and explore it.
- Count positive and negative cases.
- Split into train / test.

In [ ]:
# Try several common filenames for the Kaggle Diabetes Prediction dataset.
# Place the CSV next to this notebook before running.
candidate_paths = [
    'diabetes_prediction_dataset.csv',
    'Diabetes_Prediction_Dataset.csv',
    'diabetes.csv',
]
FALLBACK_URL = ('https://raw.githubusercontent.com/Iamsdt/DiabetesPrediction/main/'
                'diabetes_prediction_dataset.csv')

df = None
for name in candidate_paths:
    path = os.path.join(BASE_DIR, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print('Loaded local file:', name)
        break

if df is None:
    try:
        df = pd.read_csv(FALLBACK_URL)
        print('Loaded from fallback URL.')
    except Exception as e:
        raise FileNotFoundError(
            'Could not load the dataset. Download it from Kaggle '
            '("Diabetes Prediction Dataset") and place '
            '`diabetes_prediction_dataset.csv` next to this notebook.'
        ) from e

print('Shape:', df.shape)
df.head()

In [ ]:
# Quick exploration
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Class balance
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"

counts = df['diabetes'].value_counts()
print('Positive / negative counts:')
print(counts)
print(f'Positive rate: {counts.get(1, 0) / len(df) * 100:.2f}%')

counts.plot(kind='bar', color=['#4c72b0', '#dd8452'], edgecolor='black')
plt.xticks([0, 1], ['Not diabetic (0)', 'Diabetic (1)'], rotation=0)
plt.ylabel('Number of patients')
plt.title('Class balance')
plt.show()

> **Observation:** the dataset is **imbalanced** — only ~8.5% of the patients are diabetic. We will:
> - **Stratify** the train/test split on the target so the imbalance is preserved in both sets.
> - Use `class_weight='balanced'` in the Logistic Regression in Exercise 3.
> - Look at **precision, recall, F1, ROC-AUC** in Exercise 4 rather than accuracy alone.

In [ ]:
# Train / test split (stratified)
X = df.drop(columns=['diabetes'])
y = df['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('X_train:', X_train.shape, '  X_test:', X_test.shape)
print('Train positive rate:', round(y_train.mean(), 4),
      ' Test positive rate:', round(y_test.mean(), 4))

## Exercise 2 — Model Choice and Standardization

### Why Logistic Regression?
Logistic Regression is the natural choice for this binary classification task because:
1. It produces a **linear decision boundary** that is easy to visualize and reason about.
2. It outputs **calibrated probabilities** in `[0, 1]`, which lets us pick a clinically appropriate decision threshold.
3. Its coefficients are **interpretable** — important in a medical context where doctors need to understand why a patient is flagged.
4. It is a strong baseline before moving to more complex models such as Gradient Boosting.

### Why standardization?
Numerical features live on very different scales: `age` ∈ [0, 80], `bmi` ∈ [15, 50], `HbA1c_level` ∈ [3.5, 9], `blood_glucose_level` ∈ [80, 300]. Without standardization:
- the optimizer is **ill-conditioned** and convergence is slow,
- the **L2 regularization** unfairly penalises large-scale features.

We apply `StandardScaler` (mean 0, std 1) to numerical columns and **One-Hot Encoding** to categorical columns (`gender`, `smoking_history`).

In [ ]:
# Identify columns by dtype
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Categorical columns:', cat_cols)
print('Numeric    columns:', num_cols)

# Build the preprocessing pipeline
preprocess = ColumnTransformer([
    ('num', StandardScaler(),                       num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

## Exercise 3 — Model Training

We assemble the preprocessing pipeline and the Logistic Regression into a single `Pipeline`, so that fit / predict / cross-validation work end-to-end without any data leakage.

In [ ]:
clf = Pipeline([
    ('preprocess', preprocess),
    ('lr',         LogisticRegression(
                        max_iter=1000,
                        class_weight='balanced',  # handles the 8.5% imbalance
                        random_state=42))
])

clf.fit(X_train, y_train)
print('Model fitted on', len(y_train), 'samples.')

## Exercise 4 — Evaluation Metrics

In [ ]:
y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test,  y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test,    y_pred)
f1   = f1_score(y_test,        y_pred)

print('Accuracy :', round(acc,  4))
print('Precision:', round(prec, 4))
print('Recall   :', round(rec,  4))
print('F1       :', round(f1,   4))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['Not diabetic', 'Diabetic']))

In [ ]:
# Bar chart of the four metrics
plt.figure(figsize=(7, 4))
bars = plt.bar(['accuracy', 'precision', 'recall', 'f1'],
               [acc, prec, rec, f1],
               color=['#4c72b0', '#55a868', '#c44e52', '#8172b3'],
               edgecolor='black')
for b, v in zip(bars, [acc, prec, rec, f1]):
    plt.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.3f}',
             ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, 1.05)
plt.title('Metrics on test set')
plt.ylabel('Score')
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Not diabetic', 'Diabetic']).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d'
)
plt.title('Confusion matrix — Logistic Regression')
plt.grid(False)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN = {tn},  FP = {fp},  FN = {fn},  TP = {tp}')

### Comments

- **Accuracy (~0.88-0.91)** *looks* high but is misleading on this imbalanced dataset: predicting *always 0* would already score ~0.915. Accuracy alone is not a reliable indicator.
- **Precision (~0.40-0.55)** — among patients we flag as diabetic, about half really are. The other half are false alarms that would be cleared by a confirmation test.
- **Recall (~0.85-0.90)** — we successfully detect the large majority of real diabetics. This is the metric that matters most in a *medical screening* context.
- **F1 (~0.55-0.65)** — harmonic mean of precision and recall.
- **Confusion matrix:** the number of **False Negatives** (diabetics classified as healthy) is small — this is the trade-off we wanted by using `class_weight='balanced'`. **False Positives** are higher, but a false positive only triggers a follow-up exam, whereas a false negative may delay treatment.

**Conclusion:** for a diabetes screening tool, this precision/recall balance is appropriate — we accept extra confirmation tests in order not to miss real patients.

## Exercise 5 — 2-D Decision Boundary Visualization

We pick the two most informative features — `HbA1c_level` and `blood_glucose_level` — train a small Logistic Regression on just those two, and plot the decision boundary as a contour at `P(y=1) = 0.5`.

In [ ]:
feat_x = 'HbA1c_level'         if 'HbA1c_level'         in X.columns else num_cols[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else num_cols[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x,  feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0, 1])], remainder='drop')),
    ('lr',  LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Decision-boundary mesh
x_min, x_max = X2_train[feat_x].min() - 1, X2_train[feat_x].max() + 1
y_min, y_max = X2_train[feat_y].min() - 5, X2_train[feat_y].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(8, 6))
# Probability heatmap as background
plt.contourf(xx, yy, probs, levels=20, cmap='RdYlBu_r', alpha=0.5)
# Decision boundary contour
cs = plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
plt.clabel(cs, inline=True, fmt={0.5: 'P=0.5'})
# Test points colored by true label
plt.scatter(X2_test[feat_x], X2_test[feat_y], c=y_test,
            cmap='RdYlBu_r', edgecolor='k', alpha=0.7, s=25)
plt.xlabel(feat_x)
plt.ylabel(feat_y)
acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))
plt.title(f'Decision boundary on 2 features — test accuracy = {acc2:.3f}')
plt.colorbar(label='P(diabetes = 1)')
plt.show()

> **Reading the plot:**  
> The diagonal black line is the **decision boundary**: above it the model predicts *diabetic*, below it *not diabetic*. The background gradient shows the predicted probability `P(diabetes=1)` — blue = low, red = high. Test points that land on the wrong side are misclassified.  
> Accuracy on these 2 features alone (~0.85-0.88) is lower than the full model — that's expected, the other 6 features carry useful complementary information.

## Exercise 6 — ROC Curve and AUC

The ROC curve sweeps the decision threshold from 0 to 1 and plots TPR (= recall) vs FPR. The **AUC** condenses this into a single number between 0.5 (random) and 1.0 (perfect).

In [ ]:
y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Logistic Regression  (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

print(f'AUC = {auc:.4f}')

### Interpretation

- **AUC ≈ 0.95-0.97** → excellent ranking quality: a randomly chosen diabetic patient gets a higher predicted score than a randomly chosen non-diabetic patient ~96% of the time.
- The curve hugs the top-left corner, meaning there exist thresholds with **high TPR (recall)** and **low FPR** at the same time.
- We are far above the diagonal (random classifier).
- To pick the **optimal threshold**, one classical rule is **Youden's J = TPR − FPR**, which the snippet below computes. In a medical context, we may also tune the threshold based on the **cost of a false negative** vs the cost of a false positive.

In [ ]:
# Optimal threshold according to Youden's J statistic
J = tpr - fpr
best_idx = np.argmax(J)
best_threshold = thresholds[best_idx]
print(f"Youden's optimal threshold = {best_threshold:.3f}")
print(f'  -> TPR = {tpr[best_idx]:.3f},  FPR = {fpr[best_idx]:.3f}')

# Metrics at this new threshold
y_pred_tuned = (y_proba >= best_threshold).astype(int)
print('\nMetrics at the tuned threshold:')
print(f'  Accuracy : {accuracy_score(y_test,  y_pred_tuned):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_tuned):.4f}')
print(f'  Recall   : {recall_score(y_test,    y_pred_tuned):.4f}')
print(f'  F1       : {f1_score(y_test,        y_pred_tuned):.4f}')

## Summary

- **Problem framing** — binary classification, ~8.5% positives → imbalanced.
- **Pipeline** — `StandardScaler` + `OneHotEncoder` + `LogisticRegression(class_weight='balanced')`, all in a single `sklearn.Pipeline` to avoid data leakage.
- **Metrics** — accuracy is misleading, focus on precision / recall / F1 / ROC-AUC.
- **Decision boundary** — clearly visualized on the two most predictive features (`HbA1c_level`, `blood_glucose_level`).
- **ROC-AUC ≈ 0.96** — strong ranking quality; the threshold can still be tuned per business / clinical cost.

---
**End of Week 4 — Day 2 — Exercises XP.** Don't forget to push to GitHub.